# Entrenamiento de Modelos

Como ya hemos explicado en la investigación, lo que nosotros haremos será entrenar diferentes modelos haciendo uso de los datos de entrenamiento. En este notebook haremos crearemos modelos que hagan uso de los datos extraídos usando técnicas MFCC con preprocesamiento básico. De este modo, seguiremos el flujo básico de entrenamiento de modelos de ML:

1. Preprocesado de Datos. Esta parte ya fue realizada de manera previa con código python. En este caso, fue preprocesamiento básico.
2. Selección y Entrenamiento de Modelo.
    * Elegir algoritmo
    * Entrenar hiperparámetros
3. Evaluación de Resultados y Ajuste
4. Evaluación Final

## MFCC

In [1]:
import pickle
import numpy as np
import pandas as pd
import mlflow
import os
import joblib

#Tablas y Gráficos
import matplotlib.pyplot as plt
import seaborn as sns

#Preprocesado
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.base import clone

#Modelos
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, roc_curve, auc, classification_report)
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import PredefinedSplit

C:\Users\pedro\miniconda3\envs\env_audio_avanzado\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Busca el experimento (incluye los eliminados)
experimento = client.get_experiment_by_name("Clasificacion_Modelos_Clasicos_Notebooks")
print(f"ID: {experimento.experiment_id} | Estado: {experimento.lifecycle_stage}")

# Restaura
client.restore_experiment(experimento.experiment_id)

# Ahora ya puedes setearlo
mlflow.set_experiment("Clasificacion_Modelos_Clasicos_Notebooks")

ID: 3 | Estado: deleted


<Experiment: artifact_location='file:///C:/Users/pedro/git/RepositorioInvestigacionTFG/notebooks/mlruns/3', creation_time=1781536250318, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1781622426653, lifecycle_stage='active', name='Clasificacion_Modelos_Clasicos_Notebooks', tags={}, trace_location=None, workspace='default'>

In [2]:
#Cargamos base de datos de mlflow
mlflow.set_tracking_uri("sqlite:///../resultados/resultados_voces.db")
mlflow.set_experiment("Clasificacion_Modelos_Clasicos_Notebooks")

MlflowException: Cannot set a deleted experiment 'Clasificacion_Modelos_Clasicos_Notebooks' as the active experiment. You can restore the experiment, or permanently delete the experiment to create a new one.

In [ ]:
#Cargamos dataset de MFCC con Pickle
ruta_train = '../datos_entrenamiento/train_mfcc_basico1.pkl'
ruta_test = '../datos_entrenamiento/test_mfcc_basico1.pkl'

datos_train = pickle.load(open(ruta_train, 'rb'))
datos_test = pickle.load(open(ruta_test, 'rb'))

df_train = pd.DataFrame(datos_train)
df_test = pd.DataFrame(datos_test)

#Vamos a ver la información de nuestros datasets.
print("Forma del Dataset de entrenamiento:",df_train.shape)
print()
print("Encontrar datos nulos ---> ",df_train.isnull().sum())
print()
print("¿Hay valores nulos?: ", df_train.isnull().any().any())

print("Forma del Dataset de test:",df_test.shape)
print()
print("Encontrar datos nulos ---> ",df_test.isnull().sum())
print()
print("¿Hay valores nulos?: ", df_test.isnull().any().any())

In [ ]:
df_train.head()

Ahora, como se trata de un preprocesado básico, no hemos aplicado de manera previa ninguna forma de normalización de duración, Chunking o segmentación de datos. Esto representa un problema relevante, ya que los audios de entrada no tienen todos la misma longitud.

Este aspecto es crítico porque, debido a la naturaleza del algoritmo MFCC, el número de coeficientes extraídos depende directamente de la duración del audio. En consecuencia, cada muestra del conjunto de datos puede presentar un número diferente de características, lo que genera vectores de longitud variable.
Este comportamiento es incompatible con modelos de aprendizaje automático clásicos como SVM (Support Vector Machine) o RF (Random Forest), los cuales requieren que todas las muestras del conjunto de datos tengan exactamente la misma dimensionalidad de entrada. 

Por lo tanto, antes de alimentar los datos a cualquiera de estos modelos, será necesario aplicar alguna estrategia de homogeneización de la longitud de las secuencias. En este caso aplicaremos Aplanamiento o Pooling de datos. Sabemos que esto puede conllevar perdida de datos, sin embargo, es algo necesario en el caso de este preprocesamiento.

Nosotros estamos probando distintas opciones para compararlas entre ellas.

### Preprocesado MFCC

In [ ]:
#Aplicamos el Pooling.
#Calculamos la media y la desviación de cada matriz mfcc sobre el eje=1 (tiempo)
medias_train = df_train['mfccs'].apply(lambda x: np.mean(x, axis=1))
desviaciones_train = df_train['mfccs'].apply(lambda x: np.std(x, axis=1))

medias_test = df_test['mfccs'].apply(lambda x: np.mean(x, axis=1))
desviaciones_test = df_test['mfccs'].apply(lambda x: np.std(x, axis=1))

#Las pasamos a dataframes distintos:
columnas_media = [f'mfcc_media_{i+1}' for i in range(30)]
columnas_std = [f'mfcc_std_{i+1}' for i in range(30)]
df_medias_train = pd.DataFrame(medias_train.tolist(), columns=columnas_media)
df_stds_train = pd.DataFrame(desviaciones_train.tolist(), columns=columnas_std)
df_medias_test = pd.DataFrame(medias_test.tolist(), columns=columnas_media)
df_stds_test = pd.DataFrame(desviaciones_test.tolist(), columns=columnas_std)

#Recuperamos datos originales:
df_metadatos_train = df_train[['nombre_archivo', 'caja_toracica', 'grupo', 'fold']]
df_metadatos_test = df_test[['nombre_archivo', 'caja_toracica', 'grupo', 'fold']]

#Construimos los dataframes aplanados o planos (aplicando Pooling global)
df_train_plano = pd.concat([df_metadatos_train, df_medias_train, df_stds_train], axis=1)
df_test_plano = pd.concat([df_metadatos_test, df_medias_test, df_stds_test], axis=1)

print(df_train_plano.shape)
print(df_test_plano.shape)
df_train_plano.head()

Ahora, en nuestro dataset para cada entrada 64 características asignadas, de las cuales podemos extraer las 4 primeras que hacen referencia al nombre del archivo y sus clasificaciones, además del fold al que pertenecerían, que no son necesarios para las técnicas de reducción de características que vamos a aplicar, siendo estas matrices de correlación, mapas de calor y PCA. 

Estas técnicas las aplicamos ya que lo más seguro es que los valores generados al aplicar el proceso de Pooling por media y std estén muy correlacionados entre sí.

Hay que tener en cuenta que la reducción de características las hacemos en base del conjunto de datos de entrenamiento y luego lo aplicamos a los datos de test también.

In [ ]:
columnas_a_quitar = ['nombre_archivo', 'caja_toracica', 'grupo', 'fold']

X_train = df_train_plano.drop(columns=columnas_a_quitar)

#Calculamos la matriz de correlación
corr_matrix = X_train.corr()

def plot_corr(df, title):
    plt.figure(figsize=(10, 8))
    sns.heatmap(df.corr(), cmap='coolwarm', center=0)
    plt.title(f"Correlación - {title}")
    plt.show()

plot_corr(X_train,"MFCCs Aplanados")

Podemos observar como los últimos valores de desviación estándar presentan una elevada correlación positiva entre sí. De los demás coeficientes realmente no podemos decir gran cosa, ya que estos caen en valores intermedios, por lo que realmente no podemos descartar ninguno de ellos. Trabajaremos en la reducción de los mfcc_std_21 hasta mfcc_std_30.

En el caso de que observemos que encontramos varios valores de correlación entre features elevados, lo que podemos hacer es en sí aplicar tanto reducción manual de predictores como PCA. PCA recordar que nos permite en sí calcular, en base de un conjunto de predictores, otras features nuevas que son las más representativas. 
De este modo, eliminamos primero la redundancia más evidente y luego PCA puede extraer otro patrones más sutiles en un conjunto de predictores de menor tamaño.

In [ ]:
columnas_correlacionadas = [f'mfcc_std_{i}' for i in range(21, 31)]
X_correlacionado = X_train[columnas_correlacionadas]

matriz_corr_sub = X_correlacionado.corr()
for i in range(len(matriz_corr_sub.columns)):
    for j in range(i):
        col1 = matriz_corr_sub.columns[i]
        col2 = matriz_corr_sub.columns[j]
        correlacion = matriz_corr_sub.iloc[i, j]
        print(f"{col1} y {col2} -> Correlación: {correlacion:.3f}")

Aqui podemos observar todos los valores de correlación, donde observamos valores máximos de más 0.9 y otro valores que entran más en valores de correlación intermedios que podemos descartar.

Lo que haremos ahora será aplicar reducción en base de un umbral. Este umbral será 0.85.

In [ ]:
umbral = 0.85
pares_a_eliminar = set()

for i in range(len(matriz_corr_sub.columns)):
    for j in range(i):
        if abs(matriz_corr_sub.iloc[i, j]) > umbral:
            col1 = matriz_corr_sub.columns[i]
            col2 = matriz_corr_sub.columns[j]
            correlacion = matriz_corr_sub.iloc[i, j]
            print(f"{col1} y {col2} -> Correlación: {correlacion:.3f}")
            
            pares_a_eliminar.add(col1)

print("Columnas a eliminar: ")
print(f"{list(pares_a_eliminar)}. Número total = {len(pares_a_eliminar)}")

#Las eliminamos:
X_MFCC_reducido = df_train_plano.drop(columns = pares_a_eliminar)
X_MFCC_reducido_test = df_test_plano.drop(columns = pares_a_eliminar)

In [ ]:
X_MFCC_reducido.head()

In [ ]:
X_MFCC_reducido_test.head()

Ahora pasamos a aplicar PCA, debido a que seguimos teniendo una gran cantidad de features (52 sin contar nombre, caja, grupo y fold). PCA nos permite calcular los componentes principales de entre un conjunto de features. Estos componentes principales son en sí nuevas características y representarán combinaciones lineales de estos valores originales, llegando a representar X% de varianza, convirtiendose así en predictores más representativos. 

In [ ]:
#Usamos un Scaler para aplicar PCA
scaler = StandardScaler()
subset_X = X_MFCC_reducido.drop(columns = columnas_a_quitar)
X_MFCC_scaled = scaler.fit_transform(subset_X)

pca = PCA()
X_MFCC_pca = pca.fit_transform(X_MFCC_scaled)

# Ver varianza explicada. Con esto lo que buscamos es quedarnos con el número de predictores que 
#representar X% de los otros predictores.
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print(cumulative_variance)

Con esta función lo que podemos observar es el número de componentes/variables de MFCC reducido y el % de varianza que representan. Esto viene a significar el Nº de variables (ahora conocidos como componentes principales) que nos permiten explicar o representar un porcentaje de la variabilidad total de los datos.

Ahora nos toca elegir que porcentaje representa la mayoría de las variables. Si elegimos uno muy bajo podemos perder información y si elegimos uno muy alto, puede llevar a sobreajuste, además de no cumplir nuestro objetivo, que es buscar reducir el Nº de predictores. Creemos que un balance correcto entre reducción de dimensionalidad y perder la cantidad mínima de información puede ser el 85% de variabilidad para este caso en específico. Esto nos permitirá pasar de 52 features de MFCC a 14 componentes principales. Sigue siendo un numero elevado de predictores, pero considerablemente menor al que teníamos.

In [ ]:
#Nos quedamos con el número de predictores que representen el 85%
n_components = np.argmax(cumulative_variance >= 0.85) + 1
print(f"Componentes necesarios para explicar el 85%: {n_components}")

In [ ]:
#Ahora que sabemos el número de componentes con los que nos vamos a quedar, podemos obtenerlos.
#Para ello, reentrenamos PCA con el número exacto de componentes
scaler = StandardScaler()
X_train_num = X_MFCC_reducido.drop(columns = columnas_a_quitar)
X_test_num = X_MFCC_reducido_test.drop(columns = columnas_a_quitar)

X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

n_componentes = 0.85  # El valor que ya determinamos antes
pca = PCA(n_components=n_componentes, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)

#Con transform nos aseguramos de que no haya data leakage
X_test_pca = pca.transform(X_test_scaled)

n_componentes_finales = X_train_pca.shape[1]
print(f"El PCA ha seleccionado {n_componentes_finales} componentes principales.")

df_pca_train = pd.DataFrame(
    X_train_pca,
    columns=[f'MFCC_pca{i+1}' for i in range(n_componentes_finales)],
    index=X_MFCC_reducido.index
)
df_train_1 = pd.concat([X_MFCC_reducido[columnas_a_quitar], df_pca_train], axis=1)

df_pca_test = pd.DataFrame(
    X_test_pca,
    columns=[f'MFCC_pca{i+1}' for i in range(n_componentes_finales)],
    index=X_MFCC_reducido_test.index
)
df_test_1 = pd.concat([X_MFCC_reducido_test[columnas_a_quitar], df_pca_test], axis=1)

print("\nVista previa del conjunto de ENTRENAMIENTO post-pca:")
display(df_train_1.head())

In [ ]:
#Breve análisis visual de los datos con PCA.
plt.figure(figsize=(10, 8))

#Elegimos los 2 primeros MFCC por que son los que mas información aportan
sns.scatterplot(
    data=df_train_1, 
    x='MFCC_pca1', 
    y='MFCC_pca2', 
    hue='grupo', 
    style='grupo', 
    s=100,         
    alpha=0.8      
)

plt.title('Separabilidad de Pacientes tras aplicar PCA (2D) con respecto a grupo', fontsize=15, fontweight='bold')
plt.xlabel('Componente Principal 1 (MFCC_pca1)', fontsize=12)
plt.ylabel('Componente Principal 2 (MFCC_pca2)', fontsize=12)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Grupo Clínico')

plt.tight_layout()
plt.show()

In [ ]:
#Breve análisis visual de los datos con PCA.
plt.figure(figsize=(10, 8))

#Elegimos los 2 primeros MFCC por que son los que mas información aportan
sns.scatterplot(
    data=df_train_1, 
    x='MFCC_pca1', 
    y='MFCC_pca2', 
    hue='caja_toracica', 
    style='caja_toracica', 
    s=100,         
    alpha=0.8      
)

plt.title('Separabilidad de Pacientes tras aplicar PCA (2D) con respecto a caja_toracica', fontsize=15, fontweight='bold')
plt.xlabel('Componente Principal 1 (MFCC_pca1)', fontsize=12)
plt.ylabel('Componente Principal 2 (MFCC_pca2)', fontsize=12)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Caja_Toracica')

plt.tight_layout()
plt.show()

Con estos gráficos de dispersión buscamos representar cada paciente como un punto en un gráfico de coordenadas, teniendo en cuenta los valores de MFCC_pca1 y MFCC_pca2. Los 2 primeros componentes principales de un PCA son aquellas que agrupan mayor cantidad de información (varianza). En ambos gráficos podemos observar que no hay nubes de puntos claramente separadas. No hay correlación verdadera entre los datos y el tipo de enfermedad con este tipo de procesamiento, se trata de una complejidad real. Necesitaremos ver que proyectan los modelos, pero de momento no creemos que se obtengan muy buenos resultados.  

## Entrenamiento Modelos

Para este caso aplicaremos modelos básicos como SVM, RF o GB sobre los datos procesados de MFCC. Como hemos observado, los datos de momento no presentan mucha correlación con respecto al grupo o a la caja_toracica (patología). Esperamos que para este primer análisis no obtener resultados excesivamente excelentes.

In [32]:
from sklearn.metrics import make_scorer
scoring_cv = {
    'accuracy':    make_scorer(accuracy_score),
    'f1_macro':    make_scorer(f1_score, average='macro',    zero_division=0),
    'f1_weighted': make_scorer(f1_score, average='weighted', zero_division=0),
}

columnas_pca = [col for col in df_train_1.columns if 'pca' in col]

X_train_model = df_train_1[columnas_pca]
X_test_model = df_test_1[columnas_pca]

y_train_grupo = df_train_1['grupo']
y_test_grupo = df_test_1['grupo']

y_train_caja = df_train_1['caja_toracica']
y_test_caja = df_test_1['caja_toracica']

#Función común para la evaluación del modelo:
def evaluar_modelo(grid_grupo, grid_caja, nombre_modelo):    
    print(f"Mejores Hiperparámetros Grupo: {grid_grupo.best_params_}")
    print(f"Mejores Hiperparámetros Caja:  {grid_caja.best_params_}")
    idx_g = grid_grupo.best_index_
    idx_c = grid_caja.best_index_
    acc_g = grid_grupo.cv_results_['mean_test_accuracy'][idx_g]
    f1_mac_g = grid_grupo.cv_results_['mean_test_f1_macro'][idx_g]
    f1_w_g = grid_grupo.cv_results_['mean_test_f1_weighted'][idx_g]
    std_g = grid_grupo.cv_results_['std_test_f1_macro'][idx_g]
    acc_c = grid_caja.cv_results_['mean_test_accuracy'][idx_c]
    f1_mac_c = grid_caja.cv_results_['mean_test_f1_macro'][idx_c]
    f1_w_c = grid_caja.cv_results_['mean_test_f1_weighted'][idx_c]
    std_c = grid_caja.cv_results_['std_test_f1_macro'][idx_c]
    print(f"[CV] Grupo - Acc: {acc_g*100:.2f}% | F1-Macro: {f1_mac_g*100:.2f}% (+/-{std_g*100:.2f}%) | F1-Weighted: {f1_w_g*100:.2f}%")
    print(f"[CV] Caja  - Acc: {acc_c*100:.2f}% | F1-Macro: {f1_mac_c*100:.2f}% (+/-{std_c*100:.2f}%) | F1-Weighted: {f1_w_c*100:.2f}%")
    mlflow.log_metric("grupo_acc_cv", acc_g)
    mlflow.log_metric("grupo_f1_macro_cv", f1_mac_g)
    mlflow.log_metric("grupo_f1_weighted_cv", f1_w_g)
    mlflow.log_metric("caja_acc_cv", acc_c)
    mlflow.log_metric("caja_f1_macro_cv", f1_mac_c)
    mlflow.log_metric("caja_f1_weighted_cv", f1_w_c)
    for key, value in grid_grupo.best_params_.items():
        mlflow.log_param(f"grupo_best_{key}", value)
    for key, value in grid_caja.best_params_.items():
        mlflow.log_param(f"caja_best_{key}", value)

#### RF

In [33]:
#Tunearemos los hiperparámetros con las ejecuciones y veremos cuales son las mejores configuraciones para cada modelo. Para cada uno haremos que la ejecución inicial sea con 
#hiperparámetros base de la documentación

#Comenzamos con RF: comenzaremos utilizando sus hiperparámetros de base:
#https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
param_grid_rf = {
    'n_estimators': [100],      
    'max_depth': [None],        
    'min_samples_split': [2],   
    'min_samples_leaf': [1],    
    'max_features': ['sqrt'], 
    'bootstrap': [True]         
}

folds_oficiales = df_train_1['fold'].values
ps = PredefinedSplit(test_fold=folds_oficiales)
with mlflow.start_run(run_name="RF_ProcesadoBasico_PCA_default"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "Random Forest")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    rf_base_grupo = RandomForestClassifier(class_weight='balanced', random_state=42)
    grid_rf_grupo = GridSearchCV(rf_base_grupo, param_grid_rf, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_rf_grupo.fit(X_train_model, y_train_grupo)

    rf_base_caja = RandomForestClassifier(class_weight='balanced', random_state=42)
    grid_rf_caja = GridSearchCV(rf_base_caja, param_grid_rf, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_rf_caja.fit(X_train_model, y_train_caja)

    evaluar_modelo(grid_rf_grupo, grid_rf_caja, "Random Forest")
    

MlflowException: The experiment 3 must be in the 'active' state. Current state is deleted.

In [ ]:
param_grid_rf_1 = {
    'n_estimators': [75, 100, 150],      
    'max_depth': [None, 10, 20],        
    'min_samples_split': [2, 5],   
    'min_samples_leaf': [1, 2],    
    'max_features': [1.0,'sqrt'], 
    'bootstrap': [True]         
}

#Entrenamos para cada objetivo
with mlflow.start_run(run_name="RF_ProcesadoBasico_PCA_tune1"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "Random Forest")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    rf_base_grupo = RandomForestClassifier(class_weight='balanced', random_state=42)
    grid_rf_grupo = GridSearchCV(rf_base_grupo, param_grid_rf_1, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_rf_grupo.fit(X_train_model, y_train_grupo)

    rf_base_caja = RandomForestClassifier(class_weight='balanced', random_state=42)
    grid_rf_caja = GridSearchCV(rf_base_caja, param_grid_rf_1, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_rf_caja.fit(X_train_model, y_train_caja)

    evaluar_modelo(grid_rf_grupo, grid_rf_caja, "Random Forest")

In [ ]:
#Segunda configuración de Hiperparámetros.
param_grid_rf_2 = {
    'n_estimators': [100, 200],      
    'max_depth': [None],        
    'min_samples_split': [2, 5, 7],   
    'min_samples_leaf': [1,2],    
    'max_features': ['sqrt'], 
    'bootstrap': [True]         
}

#Entrenamos para cada objetivo
with mlflow.start_run(run_name="RF_ProcesadoBasico_PCA_tune2"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "Random Forest")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    rf_base_grupo = RandomForestClassifier(class_weight='balanced', random_state=42)
    grid_rf_grupo = GridSearchCV(rf_base_grupo, param_grid_rf_2, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_rf_grupo.fit(X_train_model, y_train_grupo)

    rf_base_caja = RandomForestClassifier(class_weight='balanced', random_state=42)
    grid_rf_caja = GridSearchCV(rf_base_caja, param_grid_rf_2, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_rf_caja.fit(X_train_model, y_train_caja)

    evaluar_modelo(grid_rf_grupo, grid_rf_caja, "Random Forest")

Como podemos observar, el rendimiento no puede mejorar mucho con tuneo de hiperparámetros.

#### SVM

In [ ]:
#https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
param_grid_svm = {
    'C': [1.0],
    'kernel': ['rbf'],
    'gamma': ['scale']
}

#Entrenamos para cada objetivo
with mlflow.start_run(run_name="SVM_ProcesadoBasico_PCA_default"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "SVM")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    svm_base_grupo = SVC(class_weight='balanced', random_state=42)
    grid_svm_grupo = GridSearchCV(svm_base_grupo, param_grid_svm, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_svm_grupo.fit(X_train_model, y_train_grupo)
    
    svm_base_caja = SVC(class_weight='balanced', random_state=42)
    grid_svm_caja = GridSearchCV(svm_base_caja, param_grid_svm, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_svm_caja.fit(X_train_model, y_train_caja)
    
    evaluar_modelo(grid_svm_grupo, grid_svm_caja, "SVM")


In [ ]:
param_grid_svm_1 = {
    'C': [0.5, 1, 1.5],
    'kernel': ['rbf'],
    'gamma': ['scale']
}

with mlflow.start_run(run_name="SVM_ProcesadoBasico_PCA_tune1"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "SVM")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    svm_base_grupo = SVC(class_weight='balanced', random_state=42)
    grid_svm_grupo = GridSearchCV(svm_base_grupo, param_grid_svm_1, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_svm_grupo.fit(X_train_model, y_train_grupo)
    
    svm_base_caja = SVC(class_weight='balanced', random_state=42)
    grid_svm_caja = GridSearchCV(svm_base_caja, param_grid_svm_1, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_svm_caja.fit(X_train_model, y_train_caja)
    
    evaluar_modelo(grid_svm_grupo, grid_svm_caja, "SVM")

En este caso podemos observar que el rendimiento del modelo no puede mejorar mucho más simplemente haciendo tuning de hiperparámetros. 

#### Gradient Boosting

In [34]:
#https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html
param_grid_gb = {
    'n_estimators': [100],
    'learning_rate': [0.1],
    'max_depth': [3],
    'min_samples_split': [2]
}

with mlflow.start_run(run_name="GB_ProcesadoBasico_PCA_default"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "GB")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    pesos_grupo = compute_sample_weight(class_weight='balanced', y=y_train_grupo)
    pesos_caja = compute_sample_weight(class_weight='balanced', y=y_train_caja)
    
    gb_base_grupo = GradientBoostingClassifier(random_state=42)
    grid_gb_grupo = GridSearchCV(gb_base_grupo, param_grid_gb, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_grupo.fit(X_train_model, y_train_grupo, **{'sample_weight': pesos_grupo})
    
    gb_base_caja = GradientBoostingClassifier(random_state=42)
    grid_gb_caja = GridSearchCV(gb_base_caja, param_grid_gb, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_caja.fit(X_train_model, y_train_caja, **{'sample_weight': pesos_caja})
    
    evaluar_modelo(grid_gb_grupo, grid_gb_caja, "Gradient Boosting")    


MlflowException: The experiment 3 must be in the 'active' state. Current state is deleted.

In [35]:
param_grid_gb_1 = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [1,3],
    'min_samples_split': [2, 3]
}

with mlflow.start_run(run_name="GB_ProcesadoBasico_PCA_tune1"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "GB")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    pesos_grupo = compute_sample_weight(class_weight='balanced', y=y_train_grupo)
    pesos_caja = compute_sample_weight(class_weight='balanced', y=y_train_caja)
    
    gb_base_grupo = GradientBoostingClassifier(random_state=42)
    grid_gb_grupo = GridSearchCV(gb_base_grupo, param_grid_gb_1, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_grupo.fit(X_train_model, y_train_grupo, **{'sample_weight': pesos_grupo})
    
    gb_base_caja = GradientBoostingClassifier(random_state=42)
    grid_gb_caja = GridSearchCV(gb_base_caja, param_grid_gb_1, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_caja.fit(X_train_model, y_train_caja, **{'sample_weight': pesos_caja})
    
    evaluar_modelo(grid_gb_grupo, grid_gb_caja, "Gradient Boosting")



MlflowException: The experiment 3 must be in the 'active' state. Current state is deleted.

In [36]:
param_grid_gb_2 = {
    'n_estimators': [100, 150],
    'learning_rate': [0.1, 0.15],
    'max_depth': [3,5],
    'min_samples_split': [3, 4]
}

with mlflow.start_run(run_name="GB_ProcesadoBasico_PCA_tune2"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "GB")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    pesos_grupo = compute_sample_weight(class_weight='balanced', y=y_train_grupo)
    pesos_caja = compute_sample_weight(class_weight='balanced', y=y_train_caja)
    
    gb_base_grupo = GradientBoostingClassifier(random_state=42)
    grid_gb_grupo = GridSearchCV(gb_base_grupo, param_grid_gb_2, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_grupo.fit(X_train_model, y_train_grupo, **{'sample_weight': pesos_grupo})
    
    gb_base_caja = GradientBoostingClassifier(random_state=42)
    grid_gb_caja = GridSearchCV(gb_base_caja, param_grid_gb_2, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_caja.fit(X_train_model, y_train_caja, **{'sample_weight': pesos_caja})
    
    evaluar_modelo(grid_gb_grupo, grid_gb_caja, "Gradient Boosting")

MlflowException: The experiment 3 must be in the 'active' state. Current state is deleted.

Vemos que si utilizamos muchas combinaciones obtenemos un rendimiento peor. Esto se puede deber a un sobreentrenamiento. Para ello cambiaremos el planteamiento en este caso y buscaremos no tener muchas combinaciones para los grids de hiperparámetros.

In [25]:
param_grid_gb_3_grupo = {
    'n_estimators': [150, 175],
    'learning_rate': [0.1, 0.15],
    'max_depth': [5, 7],
    'min_samples_split': [4, 6]
}

param_grid_gb_3_caja = {
    'n_estimators': [100],
    'learning_rate': [0.1],
    'max_depth': [3,5],
    'min_samples_split': [3]
}

with mlflow.start_run(run_name="GB_ProcesadoBasico_PCA_tune3"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "GB")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    pesos_grupo = compute_sample_weight(class_weight='balanced', y=y_train_grupo)
    pesos_caja = compute_sample_weight(class_weight='balanced', y=y_train_caja)
    
    gb_base_grupo = GradientBoostingClassifier(random_state=42)
    grid_gb_grupo = GridSearchCV(gb_base_grupo, param_grid_gb_3_grupo, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_grupo.fit(X_train_model, y_train_grupo, **{'sample_weight': pesos_grupo})
    
    gb_base_caja = GradientBoostingClassifier(random_state=42)
    grid_gb_caja = GridSearchCV(gb_base_caja, param_grid_gb_3_caja, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_caja.fit(X_train_model, y_train_caja, **{'sample_weight': pesos_caja})
    
    evaluar_modelo(grid_gb_grupo, grid_gb_caja, "Gradient Boosting")

Mejores Hiperparámetros Grupo: {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 4, 'n_estimators': 150}
Mejores Hiperparámetros Caja:  {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_split': 3, 'n_estimators': 100}
[CV] Grupo - Acc: 26.04% | F1-Macro: 22.34% (+/-13.26%) | F1-Weighted: 19.70%
[CV] Caja  - Acc: 25.69% | F1-Macro: 22.32% (+/-12.71%) | F1-Weighted: 20.42%


In [26]:
param_grid_gb_4_grupo = {
    'n_estimators': [150],
    'learning_rate': [0.15, 0.2],
    'max_depth': [5],
    'min_samples_split': [6, 7]
}

param_grid_gb_4_caja = {
    'n_estimators': [100, 200],
    'learning_rate': [0.1],
    'max_depth': [3],
    'min_samples_split': [3, 6]
}


with mlflow.start_run(run_name="GB_ProcesadoBasico_PCA_tune4"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "GB")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    pesos_grupo = compute_sample_weight(class_weight='balanced', y=y_train_grupo)
    pesos_caja = compute_sample_weight(class_weight='balanced', y=y_train_caja)
    
    gb_base_grupo = GradientBoostingClassifier(random_state=42)
    grid_gb_grupo = GridSearchCV(gb_base_grupo, param_grid_gb_4_grupo, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_grupo.fit(X_train_model, y_train_grupo, **{'sample_weight': pesos_grupo})
    
    gb_base_caja = GradientBoostingClassifier(random_state=42)
    grid_gb_caja = GridSearchCV(gb_base_caja, param_grid_gb_4_caja, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_caja.fit(X_train_model, y_train_caja, **{'sample_weight': pesos_caja})
    
    evaluar_modelo(grid_gb_grupo, grid_gb_caja, "Gradient Boosting")

Mejores Hiperparámetros Grupo: {'learning_rate': 0.15, 'max_depth': 5, 'min_samples_split': 6, 'n_estimators': 150}
Mejores Hiperparámetros Caja:  {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_split': 6, 'n_estimators': 100}
[CV] Grupo - Acc: 24.50% | F1-Macro: 20.31% (+/-10.11%) | F1-Weighted: 19.24%
[CV] Caja  - Acc: 27.56% | F1-Macro: 24.41% (+/-12.21%) | F1-Weighted: 21.82%


In [27]:
param_grid_gb_5_grupo = {
    'n_estimators': [150],
    'learning_rate': [0.2],
    'max_depth': [5],
    'min_samples_split': [6, 8]
}

param_grid_gb_5_caja = {
    'n_estimators': [100],
    'learning_rate': [0.1],
    'max_depth': [3],
    'min_samples_split': [6, 8]
}

with mlflow.start_run(run_name="GB_ProcesadoBasico_PCA_tune5"):

    mlflow.set_tag("tipo_dataset", "MFCC_raw") 
    mlflow.set_tag("modelo", "GB")
    mlflow.set_tag("feature_extraction", "PCA")
    
    # Entrenamos para cada objetivo
    pesos_grupo = compute_sample_weight(class_weight='balanced', y=y_train_grupo)
    pesos_caja = compute_sample_weight(class_weight='balanced', y=y_train_caja)
    
    gb_base_grupo = GradientBoostingClassifier(random_state=42)
    grid_gb_grupo = GridSearchCV(gb_base_grupo, param_grid_gb_5_grupo, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_grupo.fit(X_train_model, y_train_grupo, **{'sample_weight': pesos_grupo})
    
    gb_base_caja = GradientBoostingClassifier(random_state=42)
    grid_gb_caja = GridSearchCV(gb_base_caja, param_grid_gb_5_caja, scoring=scoring_cv, refit='f1_macro', cv=ps, n_jobs=-1)
    grid_gb_caja.fit(X_train_model, y_train_caja, **{'sample_weight': pesos_caja})
    
    evaluar_modelo(grid_gb_grupo, grid_gb_caja, "Gradient Boosting")

Mejores Hiperparámetros Grupo: {'learning_rate': 0.2, 'max_depth': 5, 'min_samples_split': 6, 'n_estimators': 150}
Mejores Hiperparámetros Caja:  {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_split': 8, 'n_estimators': 100}
[CV] Grupo - Acc: 23.34% | F1-Macro: 19.20% (+/-10.20%) | F1-Weighted: 16.71%
[CV] Caja  - Acc: 28.18% | F1-Macro: 28.25% (+/-17.75%) | F1-Weighted: 22.47%


Con esto podemos observar como ya hemos alcanzado la mejor combinación de hiperparámetros para Gradient Boosting. En general podemos concluir que utilizar MFCC con un preprocesamiento básico, sin chunking y con pooling, no nos lleva a resultados excesivamente esperenzadores. No podemos establecer que sea un fracaso, ya que como hemos dicho, nuestro objetivo es buscar alternativas y probar, sin embargo, para esta alternativa específica, los resultados han sido mediocres, incluso podemos decir que malos. Por ello, buscaremos otros métodos que puedan funcionar mejor.

## Evaluación Final: Test

De este dataset no sacamos ningún modelo